In [14]:
from src.data_generator import DataGenerator
from src.gaussian_mixture import GaussianMixtureMAR
from src.ampute_mar import AmputeMAR
from src.motivating_example import UtoM
import numpy as np
import torch
from src.metrics import energy_distance_faster

from joblib import Parallel, delayed
import matplotlib.pyplot as plt

In [ ]:
class MissingDataGenerator(DataGenerator, AmputeMAR, UtoM):

    def __init__(self, 
                 n:int, d:int,
                 distr = "normal", mu = None, sigma = None, pi = None, alpha = 0.7, c = 0.01, df = None, device = "cpu", dtype=torch.float64):
        DataGenerator.__init__(self, distr=distr, mu=mu, sigma=sigma, pi=pi, alpha=alpha, c=c, df=df, device=device, dtype=dtype)
        AmputeMAR.__init__(self)
        UtoM.__init__(self, d=d)
        self.n = n
        self.d = d

    def get_X(self):
        return self.sample_truth(n=self.n, d=self.d)

    def get_M_ampute(self, X, n_pat, prop=None, seed=123):
        rng = np.random.default_rng(seed)
        patterns = self.sample_patterns(self.d, self.n, npattern=n_pat, rng=rng)
        Xmis, M = self.ampute(X, patterns=patterns, prop=prop)
        return Xmis, M


    def get_M_eps_contaminated_mar(self, X, eps):
        u1 = self.x_to_u(X[:,0])
        u2 = self.x_to_u(X[:,1])
        M = self.get_M(missing_type='contaminated_mar', u1=u1, u2=u2, eps=eps)
        return M

    def get_M_ampute_eps_contaminated(self, X, eps, c0, n_pat, prop=None, seed=123):
        rng = np.random.default_rng(seed)
        patterns = self.sample_patterns(self.d, self.n, npattern=n_pat, rng=rng)
        complete = rng.random(n) < c0
        mnar = ~complete & (rng.random(n) < eps / (1 - c0))   # gives eps*n MNAR rows on average
        mar = ~complete & ~mnar
    
        out = X.to_numpy(dtype=float).copy()
        for mask, mech, s in [(mnar, "MNAR", seed), (mar, "MAR", seed + 1)]:
            if mask.sum() == 0:
                continue
            out[mask] = self.ampute(X, patterns=patterns, prop=prop)

        return torch.tensor(out, device=self.device, dtype=self.dtype)


In [ ]:
import torch
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

from src.converter import Converter


# import os
# from pathlib import Path
# R_HOME = Path(r"C:\Users\hu\AppData\Local\Programs\R\R-4.5.3")
# assert (R_HOME / "bin" / "R.exe").exists(), "R.exe not found"
# assert (R_HOME / "bin" / "x64" / "R.dll").exists(), "R.dll not found"
# os.environ["R_HOME"] = str(R_HOME)
# os.environ["PATH"] = (
#     str(R_HOME / "bin" / "x64")
#     + os.pathsep
#     + str(R_HOME / "bin")
#     + os.pathsep
#     + os.environ["PATH"]
# )


mice = importr("mice")

class AmputeMAR(Converter):

    @staticmethod
    def sample_patterns(d, n, npattern=None, rng=None):
        """Sample incomplete missingness patterns once (1 = observed, 0 = missing).
        Last column is always observed; all-ones rows are excluded."""
        rng = np.random.default_rng() if rng is None else rng
        while True:
            if npattern is None:
                npattern = int(rng.integers(3, max(round(n / 100), 3) + 1))
            #npattern= 10
            patterns = rng.integers(0, 2, size=(npattern, d - 1))
            patterns = np.hstack([patterns, np.ones((npattern, 1), dtype=int)])
            incomplete = patterns[~patterns.all(axis=1)]
            if incomplete.shape[0] > 0:
                return incomplete

    def ampute(self, X, patterns=None, prop=None, rng=None, miss_type=None, weights=None):
        """
        Parameters
        ----------
        X : torch.Tensor, shape (n, d)
            Complete data.
        patterns : array-like, shape (n_pats, d), optional
            Binary patterns: 1 = observed, 0 = missing. All-ones rows are dropped.
            If None, new patterns are sampled.
        prop : float, optional
            Proportion of cases with missing values (so 1 - prop stay complete).
            Default: n_pats / (n_pats + 1), i.e. the complete cases are as
            frequent as each incomplete pattern.
        rng : np.random.Generator, optional
            Only used if patterns have to be sampled.

        Returns
        -------
        X_NA : torch.Tensor
            Data with NaN values.
        M : torch.Tensor
            Observation mask: 1 = observed, 0 = missing.
        """
        n, d = X.shape

        if patterns is None:
            patterns = self.sample_patterns(d, n, rng)
        patterns = np.asarray(patterns, dtype=int)
        patterns = patterns[~patterns.all(axis=1)]
        n_pats = patterns.shape[0]

        if prop is None:
            prop = n_pats / (n_pats + 1)

        if n_pats > 0:
            result = mice.ampute(
                self.to_r(X),
                patterns=self.to_r(patterns),
                prop=prop,              # freq defaults to equal across patterns
                mech="MAR",
                bycases=True,
                type=self.to_r(miss_type),
                weights=self.to_r(weights)
            )
            amp = self.to_python(result.rx2("amp"))
            amp = amp.to_numpy() if hasattr(amp, "to_numpy") else np.asarray(amp)
            X_NA = torch.tensor(amp, device=X.device, dtype=X.dtype)   # (n, d)

        M = ~torch.isnan(X_NA)

        return X_NA, M

    def mixed_ampute(X, c0, eps, pattern_defs, seed=0):
        rng = np.random.default_rng(seed)
        n = len(X)
        
        patterns_df, freq_list = build_patterns(X.columns, pattern_defs)
        
        complete = rng.random(n) < c0
        mnar = ~complete & (rng.random(n) < eps / (1 - c0))   # gives eps*n MNAR rows on average
        mar = ~complete & ~mnar

        out = X.to_numpy(dtype=float).copy()
        for mask, mech, s in [(mnar, "MNAR", seed), (mar, "MAR", seed + 1)]:
            if mask.sum() == 0:
                continue
            out[mask] = r_ampute(X[mask], mech, s, patterns_df, freq_list)

        X_amp = pd.DataFrame(out, columns=X.columns, index=X.index)
        return X_amp, complete, mnar, mar

In [ ]:
def build_patterns(columns, pattern_defs):
    """mice::ampute convention: 0 = set to missing, 1 = stays observed."""
    rows, freqs = [], []
    for missing_cols, freq in pattern_defs:
        missing = set(missing_cols)
        rows.append([0 if c in missing else 1 for c in columns])
        freqs.append(freq)
    patterns_df = pd.DataFrame(rows, columns=list(columns))
    return patterns_df, freqs

pattern_defs = [
    (["mean radius", "mean perimeter", "mean area"], 0.15),
    (["mean compactness", "mean concavity", "mean concave points"], 0.15),
    ([c for c in cols if c.startswith("worst")], 0.20),
    ([c for c in cols if c.endswith("error")], 0.20),
    ([c for c in cols if "radius" in c] + [c for c in cols if "area" in c] + [c for c in cols if "perimeter" in c], 0.2),
    (["target"], 0.1)
]

In [19]:
import pandas as pd

def build_patterns(columns, pattern_defs):
    """mice::ampute convention: 0 = set to missing, 1 = stays observed."""
    rows, freqs = [], []
    for missing_cols, freq in pattern_defs:
        missing = set(missing_cols)
        rows.append([0 if c in missing else 1 for c in columns])
        freqs.append(freq)
    patterns_df = pd.DataFrame(rows, columns=list(columns))
    return patterns_df, freqs

def r_ampute(df, mech, seed, patterns_df, freq_list):
    """Call mice::ampute on a pandas DataFrame, return a numpy array with NaNs."""
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr

    mice = importr("mice")   # cached by rpy2, so repeated calls are cheap

    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(df)
        r_patterns = ro.conversion.py2rpy(patterns_df)
    r_freq = ro.FloatVector(freq_list)

    ro.r("set.seed")(int(seed))
    res = mice.ampute(r_df, prop=0.999, patterns=r_patterns, freq=r_freq, mech=mech)
    amp = res.rx2("amp")
    with localconverter(ro.default_converter + pandas2ri.converter):
        return ro.conversion.rpy2py(amp).to_numpy(dtype=float)


def mixed_ampute(X, c0, eps, pattern_defs, seed=0):
    rng = np.random.default_rng(seed)
    n = len(X)
    
    patterns_df, freq_list = build_patterns(X.columns, pattern_defs)
    
    complete = rng.random(n) < c0
    mnar = ~complete & (rng.random(n) < eps / (1 - c0))   # gives eps*n MNAR rows on average
    mar = ~complete & ~mnar

    out = X.to_numpy(dtype=float).copy()
    for mask, mech, s in [(mnar, "MNAR", seed), (mar, "MAR", seed + 1)]:
        if mask.sum() == 0:
            continue
        out[mask] = r_ampute(X[mask], mech, s, patterns_df, freq_list)

    X_amp = pd.DataFrame(out, columns=X.columns, index=X.index)
    return X_amp, complete, mnar, mar

In [ ]:
# pattern definitions: columns that get set to missing, plus frequency
pattern_defs = [
    (["mean radius", "mean perimeter", "mean area"], 0.15),
    (["mean compactness", "mean concavity", "mean concave points"], 0.15),
    ([c for c in cols if c.startswith("worst")], 0.20),
    ([c for c in cols if c.endswith("error")], 0.20),
    ([c for c in cols if "radius" in c] + [c for c in cols if "area" in c] + [c for c in cols if "perimeter" in c], 0.2),
    (["target"], 0.1)
]

mixed_ampute(X, 0.1, 0.1, )

In [ ]:
def mixed_ampute(X, c0, eps, pattern_defs, seed=0):
    rng = np.random.default_rng(seed)
    n = len(X)
    
    patterns_df, freq_list = build_patterns(X.columns, pattern_defs)
    
    complete = rng.random(n) < c0
    mnar = ~complete & (rng.random(n) < eps / (1 - c0))   # gives eps*n MNAR rows on average
    mar = ~complete & ~mnar

    out = X.to_numpy(dtype=float).copy()
    for mask, mech, s in [(mnar, "MNAR", seed), (mar, "MAR", seed + 1)]:
        if mask.sum() == 0:
            continue
        out[mask] = r_ampute(X[mask], mech, s, patterns_df, freq_list)

    X_amp = pd.DataFrame(out, columns=X.columns, index=X.index)
    return X_amp, complete, mnar, mar

In [3]:
n = 2000
d = 3

In [4]:
mdg = MissingDataGenerator(distr='logistic', n=n, d=d)

In [ ]:
def job(
        k_range:tuple = None,
        eps:int = None
):
    X = mdg.get_X()
    M = mdg.get_M_ampute(X, eps=eps)
    Xtest = mdg.get_X()
    model = GaussianMixtureMAR(k_range=k_range)
    model.fit(X, M)
    Xtilde = model.sample(n)[0]
    return energy_distance_faster(Xtest, Xtilde, scale=True).item()

In [17]:
k_range = (0,20)
eps_vals = [0.0, 0.1, 0.2, 0.5, 0.7, 0.9]
B = 20

In [18]:
results = np.array(
    Parallel(n_jobs=15, backend="threading")(
    delayed(job)(k_range=k_range, eps=eps)
    for eps in eps_vals
    for _ in range(B)
)
)

KeyboardInterrupt: 

In [ ]:
plt.boxplot(results)
plt.xlabel('$\varepsilon$')
plt.ylabel('Energy Distance')
plt.show()